In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

In [5]:
pdfLoader = PyPDFLoader("../data/Sri_krishna_medical_report_medibuddy.pdf")
docs = pdfLoader.load()
len(docs)
docs[0].page_content[:500]

'Name : MR. SRI KRISHNA KOLLIPARA\nAge/Gender : 21 years / Male\nSample Type : WB EDTA\nSample ID : 24220622\nClient Name : 3KLTVM1831\nRef. Doctor : SELF\nCollected : Mar 07, 2025, 02:56 p.m.\nReceived : Mar 07, 2025, 08:18 p.m.\nReported : Mar 07, 2025, 11:56 p.m.\nMEDID :MB1457286\nCLINICAL BIOCHEMISTRY\nTEST DESCRIPTION RESULT UNITS REFERENCE RANGES\n This is an electronically authenticated report. Report Printed Date: Mar 08, 2025, 05:16 a.m.\nNOTE: Assay results should be correlated clinically with othe'

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = splitter.split_documents(docs)
split_docs[0].page_content

'Name : MR. SRI KRISHNA KOLLIPARA\nAge/Gender : 21 years / Male\nSample Type : WB EDTA\nSample ID : 24220622\nClient Name : 3KLTVM1831\nRef. Doctor : SELF\nCollected : Mar 07, 2025, 02:56 p.m.\nReceived : Mar 07, 2025, 08:18 p.m.\nReported : Mar 07, 2025, 11:56 p.m.\nMEDID :MB1457286\nCLINICAL BIOCHEMISTRY\nTEST DESCRIPTION RESULT UNITS REFERENCE RANGES\n This is an electronically authenticated report. Report Printed Date: Mar 08, 2025, 05:16 a.m.\nNOTE: Assay results should be correlated clinically with other clinical findings and the total clinical status of the patient.\nTCS WELLNESS PACKAGE ONSITE\nGlycosylated Hemoglobin (GHb/HBA1c)\n HbA1c\n(Method: HPLC)\n5.54 % < 6.0 : Non Diabetic\n6.1 – 6.5 : Prediabetic\n6.6 – 7.0 : Good Control\n7.1-8.0 : POOR Control\n>8.1 : ALERT\n Estimated Average Glucose (eAG) 112.30 % 70 - 136\nInterpretation:\nExcellent Control: 6  to 7  % \nFair to Good Control: 7  to 8  % \nUnsatisfactory Control: 8  to 1 0  % \nand Poor Control: More than 1 0 % .'

In [7]:
embedding_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_store = Chroma.from_documents(
    documents = split_docs, 
    embedding = embedding_model
)

In [8]:
query = "What is this abt?"
response = vector_store.similarity_search(query)
response[0].page_content

'Interpretation:\nBlood glucose test may be used to detect high blood glucose (hyperglycemia) and low blood glucose (hypoglycemia).Screen for diabetes in people who are at risk before signs\nand symptoms are apparent; in some cases, there may be no early signs or symptoms of diabetes. Screening can therefore be useful in helping to identify it and allowing for\ntreatment before the condition worsens or complications arise,Help diagnose diabetes, prediabetes and gestational diabetes,Monitor glucose levels in people diagnosed with\ndiabetes.\n**END OF REPORT**\nPage 2 of 11'

In [9]:
context = ""
for res in response:
    context += res.page_content + "\n"

print(context)

Interpretation:
Blood glucose test may be used to detect high blood glucose (hyperglycemia) and low blood glucose (hypoglycemia).Screen for diabetes in people who are at risk before signs
and symptoms are apparent; in some cases, there may be no early signs or symptoms of diabetes. Screening can therefore be useful in helping to identify it and allowing for
treatment before the condition worsens or complications arise,Help diagnose diabetes, prediabetes and gestational diabetes,Monitor glucose levels in people diagnosed with
diabetes.
**END OF REPORT**
Page 2 of 11
I aVR V1 V4
II aVL V2 V5
III aVF V3 V6
II
Speed: 25 mm/sec        F: 0.05 - 40 Hz       Limb: 10 mm/mV      Chest: 10 mm/mV
Date: IST: 2025-03-07 10:18:18 Report ID: MDA_900206920_V73I0SMI
Personal Details
UHID: 900206920
PatientID: 900206920
Name: Sri Krishna Kollipara
Age: 21
Gender: Male
Mobile: 0000000000
|
|
|
|
|
|
|
|
|
|
Pre-Existing Medical-
Conditions
|
|
|
|
|
|
|
|
|
|
Symptoms |
|
|
|
|
|
|
|
|
|
Vitals |
|
|
|


In [10]:
def get_context(query):
    response = vector_store.similarity_search(query)
    context = ""
    for res in response:
        context += res.page_content + "\n"
    return {"context": context, "question": query}

In [11]:
model = ChatGroq(model="openai/gpt-oss-20b")

In [18]:
prompt_template = ChatPromptTemplate.from_template("""
        You are a helpful assistant and provide answers based on the context for user question. and 
        if you don't know the answer, then you can say that 'I dont know.'
        Context: {context}
        Question: {question}
    """)

In [19]:
rag_chain = get_context | prompt_template | model

In [20]:
response = rag_chain.invoke("Who is PM of India?")
print(response.content)

I don't know.
